# Content Pipeline

Runs the production JSON/dict applications from collection through authoritative editorial selection, roundup scripting, and YouTube metadata.
Voiceover, ChatGPT handoffs, and visual production have separate notebooks.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
print(PROJECT_ROOT)
print(PROJECT_ROOT.parent.parent)

if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if not (PROJECT_ROOT / "src" / "pipeline" / "content").exists():
    raise RuntimeError("Run this notebook from the repository root or output directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.config import ConfigRepository
from src.pipeline.orchestration import create_run_context, module_output_dir

CONTENT_DIR = PROJECT_ROOT / "src" / "pipeline" / "content"
PIPELINE_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline" / "config" / "workbench.yaml"
OUTPUT_ROOT = PROJECT_ROOT / "output"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Content dir:  {CONTENT_DIR}")
print(f"Output root:  {OUTPUT_ROOT}")
print(f"Config:       {PIPELINE_CONFIG_PATH}")


In [ ]:
# Set RUN_PROFILE to None to use workbench.yaml's default.
# Set RESUME_RUN_ID only when intentionally continuing an existing run.
RUN_PROFILE = None
RESUME_RUN_ID = None
METADATA_SCRIPT_RESULT_PATH = None  # Optional canonical ChatGPT script result.

CONFIG_REPOSITORY = ConfigRepository(PIPELINE_CONFIG_PATH)
RUN_CONTEXT = create_run_context(
    project_root=PROJECT_ROOT,
    run_profile=RUN_PROFILE,
    run_id=RESUME_RUN_ID,
    trigger="notebook",
    orchestrator="notebooks/production/content_pipeline.ipynb",
    config_path=PIPELINE_CONFIG_PATH,
    output_root=OUTPUT_ROOT,
)
RUN_ID = RUN_CONTEXT["run_id"]
RUN_PROFILE = RUN_CONTEXT["run_profile"]
RUN_DIR = Path(RUN_CONTEXT["run_dir"])
FROZEN_PROFILE_CONFIG = deepcopy(RUN_CONTEXT["profile_config"])
PROFILE_CONFIG = CONFIG_REPOSITORY.profile(RUN_PROFILE).value

NEWS_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "news_collection")
EVIDENCE_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "evidence_prep")
EDITORIAL_TRIAGE_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "editorial_triage")
STORY_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "story_mapping")
EDITORIAL_SELECTION_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "editorial_selection")
VIDEO_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "video_packaging")
SCRIPT_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "script_generation")
METADATA_OUTPUT_DIR = module_output_dir(RUN_CONTEXT, "metadata_creator")

print(f"Run profile: {RUN_PROFILE}")
print(f"Run ID:      {RUN_ID}")
print(f"Run dir:     {RUN_DIR}")
print(
    "LLMs:        "
    f"editorial_triage={PROFILE_CONFIG['editorial_triage']['provider']}/{PROFILE_CONFIG['editorial_triage']['model']}, "
    f"story_mapping={PROFILE_CONFIG['story_mapping']['story_label']['provider']}/{PROFILE_CONFIG['story_mapping']['story_label']['model']}, "
    f"editorial_selection={PROFILE_CONFIG['editorial_selection']['provider']}/{PROFILE_CONFIG['editorial_selection']['model']}, "
    f"script_generation={PROFILE_CONFIG['script_generation']['provider']}/{PROFILE_CONFIG['script_generation']['model']}, "
    f"metadata_creator={PROFILE_CONFIG['metadata_creator']['provider']}/{PROFILE_CONFIG['metadata_creator']['model']}"
)


In [ ]:
def workbench_module_config(module_name: str) -> dict:
    stage = "ingestion" if module_name == "news_collection" else "content"
    config = CONFIG_REPOSITORY.module(RUN_PROFILE, stage, module_name).value
    if not config:
        raise KeyError(f"Profile {RUN_PROFILE!r} is missing config for {module_name!r}")
    return config


def workbench_shared_config() -> dict:
    return CONFIG_REPOSITORY.shared(RUN_PROFILE).value


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def load_json(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return payload


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    if text:
        text += "\n"
    path.write_text(text, encoding="utf-8")


def require_app_ready(result: dict, app_name: str, *, allowed_statuses=("ok",), handoff_key=None) -> None:
    status = result.get("status")
    if status not in allowed_statuses:
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or result.get("reason") or "No failure reason provided."
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")
    if handoff_key and not bool(result.get("handoff", {}).get(handoff_key, False)):
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or f"handoff.{handoff_key} is false"
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")


def require_non_empty_script(script_generation_result: dict) -> str:
    require_app_ready(script_generation_result, "script_generation", handoff_key="ready_for_metadata_creation")
    script_package = script_generation_result.get("data", {}).get("script_package", {})
    full_script = str(script_package.get("full_script", "")).strip()
    if not full_script:
        checks = script_package.get("checks", {})
        errors = checks.get("errors") or []
        reason = "; ".join(str(error) for error in errors) or "script_package.full_script is empty"
        raise RuntimeError(f"script_generation did not produce a non-empty full_script. reason={reason}")
    sections = script_package.get("sections", [])
    if not sections or any(not section.get("beats") for section in sections):
        raise RuntimeError("script_generation did not produce canonical section-aware beats")
    return full_script


def collection_window_for_profile(profile: dict) -> dict:
    end = datetime.now(IST)
    lookback_hours = int(profile["lookback_hours"])
    start = end - timedelta(hours=lookback_hours)
    return {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "lookback_hours": lookback_hours,
    }


def build_news_collection_input(run_id: str, run_profile: str) -> dict:
    authoring = workbench_module_config("news_collection")
    profile = authoring["profile"]
    source_groups = authoring["source_groups"]
    window = collection_window_for_profile(profile)
    collectors = {}
    for group_name in profile["enabled_source_groups"]:
        collector_config = deepcopy(source_groups[group_name])
        collector_config["source_group"] = group_name
        collector_config["collection_window"] = window
        collector_config["query_themes"] = list(profile["query_themes"])
        collector_config["runtime"] = deepcopy(profile.get("collector_caps", {}).get(group_name, {}))
        collectors[group_name] = collector_config

    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "enabled_source_groups": list(profile["enabled_source_groups"]),
            "query_themes": list(profile["query_themes"]),
            "collectors": collectors,
        },
        "input": {
            "run_profile": run_profile,
            "channel": deepcopy(workbench_shared_config()["channel"]),
        },
    }


def build_evidence_prep_input(run_id: str, news_input: dict, news_result: dict) -> dict:
    first_collector = next(iter(news_input["config"]["collectors"].values()))
    config = workbench_module_config("evidence_prep")
    config["collection_window"] = deepcopy(first_collector["collection_window"])
    config["query_themes"] = list(news_input["config"]["query_themes"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "news_collection": news_result,
        },
    }


def build_editorial_triage_input(run_id: str, news_result: dict, evidence_result: dict) -> dict:
    config = workbench_module_config("editorial_triage")
    config["channel"] = deepcopy(workbench_shared_config()["channel"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "news_collection": news_result,
            "evidence_prep": evidence_result,
        },
    }


def build_story_mapping_input(run_id: str, evidence_input: dict, evidence_result: dict, editorial_triage_result: dict) -> dict:
    config = workbench_module_config("story_mapping")
    config["collection_window"] = deepcopy(evidence_input["config"]["collection_window"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "evidence_prep": evidence_result,
            "editorial_triage": editorial_triage_result,
        },
    }


def build_editorial_selection_input(run_id: str, story_mapping_result: dict) -> dict:
    config = workbench_module_config("editorial_selection")
    config["channel"] = deepcopy(workbench_shared_config()["channel"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "story_mapping": story_mapping_result,
        },
    }


def build_video_packaging_input(run_id: str, editorial_selection_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": workbench_module_config("video_packaging"),
        "input": {
            "editorial_selection": editorial_selection_result,
        },
    }


def build_script_generation_input(run_id: str, video_packaging_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": workbench_module_config("script_generation"),
        "input": {
            "video_packaging": video_packaging_result,
        },
    }


def build_metadata_creator_input(run_id: str, script_generation_result: dict, video_packaging_result: dict) -> dict:
    channel = deepcopy(workbench_shared_config()["channel"])
    config = workbench_module_config("metadata_creator")
    config["channel"] = {
        "channel_name": channel["channel_name"],
        "content_promise": channel.get("content_promise", ""),
    }
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "script_generation": script_generation_result,
            "video_packaging": video_packaging_result,
        },
    }


## App 1: News Collection


In [ ]:
from src.pipeline.ingestion.news_collection import run as run_news_collection

news_input = build_news_collection_input(RUN_ID, RUN_PROFILE)
news_result = run_news_collection(news_input)
require_app_ready(news_result, "news_collection", allowed_statuses=("ok", "partial"))
write_json(NEWS_OUTPUT_DIR / "news_collection_input.json", news_input)

write_json(NEWS_OUTPUT_DIR / "news_collection_result.json", news_result)
write_json(NEWS_OUTPUT_DIR / "raw_records.json", news_result["data"])
write_json(NEWS_OUTPUT_DIR / "candidate_ledger.json", {"candidates": news_result["data"]["candidate_ledger"]})
write_json(NEWS_OUTPUT_DIR / "summary.json", news_result["summary"])

print(f"News collection status: {news_result['status']}")
print(f"Raw records:            {news_result['data']['record_count']}")
print(f"Audited candidates:     {news_result['data']['candidate_count']}")
print(f"Wrote:                  {NEWS_OUTPUT_DIR}")


In [ ]:
news_result["summary"]


## App 2: Evidence Prep


In [ ]:
from src.pipeline.content.evidence_prep import run as run_evidence_prep

evidence_input = build_evidence_prep_input(RUN_ID, news_input, news_result)
evidence_result = run_evidence_prep(evidence_input)
require_app_ready(evidence_result, "evidence_prep")
write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_input.json", evidence_input)

write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_result.json", evidence_result)
write_json(EVIDENCE_OUTPUT_DIR / "prepared_records.json", {"record_count": evidence_result["data"]["record_count"], "records": evidence_result["data"]["records"]})
write_json(EVIDENCE_OUTPUT_DIR / "decision_ledger.json", {"decisions": evidence_result["data"]["decision_ledger"]})
write_json(EVIDENCE_OUTPUT_DIR / "summary.json", evidence_result["summary"])

print(f"Evidence prep status: {evidence_result['status']}")
print(f"Prepared records:     {evidence_result['data']['record_count']}")
print(f"Prepared candidates:  {evidence_result['data']['record_count']}")
print(f"Wrote:                {EVIDENCE_OUTPUT_DIR}")


In [ ]:
evidence_result["summary"]


## Editorial Triage

Semantically evaluates all retained candidates and authoritatively chooses which records advance into story mapping.


In [ ]:
from src.pipeline.content.editorial_triage import run as run_editorial_triage

editorial_triage_input = build_editorial_triage_input(RUN_ID, news_result, evidence_result)
write_json(EDITORIAL_TRIAGE_OUTPUT_DIR / "editorial_triage_input.json", editorial_triage_input)
editorial_triage_result = run_editorial_triage(editorial_triage_input)
require_app_ready(editorial_triage_result, "editorial_triage", handoff_key="ready_for_story_mapping")

write_json(EDITORIAL_TRIAGE_OUTPUT_DIR / "editorial_triage_result.json", editorial_triage_result)
write_json(EDITORIAL_TRIAGE_OUTPUT_DIR / "decisions.json", {"decisions": editorial_triage_result.get("data", {}).get("decisions", [])})
write_json(EDITORIAL_TRIAGE_OUTPUT_DIR / "summary.json", editorial_triage_result["summary"])
write_jsonl(EDITORIAL_TRIAGE_OUTPUT_DIR / "llm_calls.jsonl", editorial_triage_result.get("llm_calls", []))

print(f"Editorial triage status: {editorial_triage_result['status']}")
print(f"Authoritative decisions: {editorial_triage_result['summary'].get('decision_count', 0)}")
print(f"Authoritative:          {editorial_triage_result.get('handoff', {}).get('authoritative', False)}")
print(f"Wrote:                  {EDITORIAL_TRIAGE_OUTPUT_DIR}")


In [ ]:
editorial_triage_result["summary"]


## App 3: Story Mapping


In [ ]:
# Optional story-mapping resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "run_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(RUN_DIR / "evidence_prep" / "evidence_prep_result.json", encoding="utf-8") as f:
#     evidence_result = json.load(f)


In [ ]:
from src.pipeline.content.story_mapping import run as run_story_mapping

story_mapping_input = build_story_mapping_input(RUN_ID, evidence_input, evidence_result, editorial_triage_result)
story_mapping_result = run_story_mapping(story_mapping_input)
require_app_ready(story_mapping_result, "story_mapping", handoff_key="ready_for_editorial_selection")
write_json(STORY_OUTPUT_DIR / "story_mapping_input.json", story_mapping_input)

write_json(STORY_OUTPUT_DIR / "story_mapping_result.json", story_mapping_result)
write_json(STORY_OUTPUT_DIR / "stories.json", story_mapping_result["data"])
write_json(STORY_OUTPUT_DIR / "summary.json", story_mapping_result["summary"])
write_jsonl(STORY_OUTPUT_DIR / "llm_calls.jsonl", story_mapping_result["llm_calls"])

print(f"Story mapping status: {story_mapping_result['status']}")
print(f"Stories:              {story_mapping_result['data']['story_count']}")
print(f"Wrote:                {STORY_OUTPUT_DIR}")


In [ ]:
story_mapping_result["summary"]


## App 4: Editorial Selection

Uses semantic editorial judgment to choose, order, and assign roles to the roundup slate. Deterministic packaging does not re-rank these decisions.


In [ ]:
from src.pipeline.content.editorial_selection import run as run_editorial_selection

editorial_selection_input = build_editorial_selection_input(RUN_ID, story_mapping_result)
editorial_selection_result = run_editorial_selection(editorial_selection_input)
require_app_ready(editorial_selection_result, "editorial_selection", handoff_key="ready_for_video_packaging")
write_json(EDITORIAL_SELECTION_OUTPUT_DIR / "editorial_selection_input.json", editorial_selection_input)
write_json(EDITORIAL_SELECTION_OUTPUT_DIR / "editorial_selection_result.json", editorial_selection_result)
write_json(EDITORIAL_SELECTION_OUTPUT_DIR / "editorial_slate.json", editorial_selection_result["data"]["editorial_slate"])
write_json(EDITORIAL_SELECTION_OUTPUT_DIR / "summary.json", editorial_selection_result["summary"])
write_jsonl(EDITORIAL_SELECTION_OUTPUT_DIR / "llm_calls.jsonl", editorial_selection_result.get("llm_calls", []))

print(f"Editorial selection status: {editorial_selection_result['status']}")
print(f"Selected stories:           {editorial_selection_result['summary']['selected_story_count']}")
print(f"Episode shape:              {editorial_selection_result['summary'].get('episode_shape', '')}")
print(f"Wrote:                      {EDITORIAL_SELECTION_OUTPUT_DIR}")


In [ ]:
editorial_selection_result["summary"]


## App 5: Video Packaging


In [ ]:
# Optional video-packaging resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "run_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(RUN_DIR / "editorial_selection" / "editorial_selection_result.json", encoding="utf-8") as f:
#     editorial_selection_result = json.load(f)


In [ ]:
from src.pipeline.content.video_packaging import run as run_video_packaging

video_packaging_input = build_video_packaging_input(RUN_ID, editorial_selection_result)
video_packaging_result = run_video_packaging(video_packaging_input)
require_app_ready(video_packaging_result, "video_packaging", handoff_key="ready_for_script_generation")
write_json(VIDEO_OUTPUT_DIR / "video_packaging_input.json", video_packaging_input)

write_json(VIDEO_OUTPUT_DIR / "video_packaging_result.json", video_packaging_result)
write_json(VIDEO_OUTPUT_DIR / "video_packet.json", video_packaging_result["data"]["video_packet"])
write_json(VIDEO_OUTPUT_DIR / "summary.json", video_packaging_result["summary"])

print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {video_packaging_result['summary']['selected_story_count']}")
print(f"Ready for script gen:   {video_packaging_result['handoff']['ready_for_script_generation']}")
print(f"Wrote:                  {VIDEO_OUTPUT_DIR}")


In [ ]:
video_packaging_result["summary"]


## App 6: Script Generation


In [ ]:
# When RESUME_RUN_ID was set in the setup cell, reuse its completed App 5 packet.
if RESUME_RUN_ID:
    video_packaging_result = load_json(RUN_DIR / "video_packaging" / "video_packaging_result.json")
    print(f"Loaded App 5 packet: {RUN_DIR / 'video_packaging' / 'video_packaging_result.json'}")


In [ ]:
from src.pipeline.content.script_generation.formats.roundup import run as run_roundup_script_generation

script_generation_input = build_script_generation_input(RUN_ID, video_packaging_result)
# Existing run contexts keep their frozen concurrency. Uncomment this only when an API tier needs serialized calls.
script_generation_input["config"]["worker_count"] = 1
script_generation_input["config"]

In [ ]:
if "script_generation_input" not in globals():
    raise RuntimeError("Run the App 5 input/config cell immediately above before this cell.")
write_json(SCRIPT_OUTPUT_DIR / "script_generation_input.json", script_generation_input)
script_generation_result = run_roundup_script_generation(script_generation_input)
script_package = script_generation_result["data"]["script_package"]

write_json(SCRIPT_OUTPUT_DIR / "script_generation_result.json", script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_package.json", script_package)
write_json(SCRIPT_OUTPUT_DIR / "summary.json", script_generation_result["summary"])
write_jsonl(SCRIPT_OUTPUT_DIR / "llm_calls.jsonl", script_generation_result["llm_calls"])
full_script = str(script_package.get("full_script", "")).strip()
if full_script:
    (SCRIPT_OUTPUT_DIR / "full_script.txt").parent.mkdir(parents=True, exist_ok=True)
    (SCRIPT_OUTPUT_DIR / "full_script.txt").write_text(full_script, encoding="utf-8")
require_non_empty_script(script_generation_result)

print(f"Script generation status: {script_generation_result['status']}")
print(f"Sections:                 {script_generation_result['summary']['section_count']}")
print(f"Audiovisual beats:        {script_package['full_script_stats']['beat_count']}")
print(f"Audio units:              {len(script_package['audio_plan']['units'])}")
print(f"Words:                    {script_generation_result['summary']['total_word_count']}")
print(f"Estimated seconds:        {script_generation_result['summary']['estimated_duration_seconds']}")
print(f"Ready for metadata:       {script_generation_result['handoff']['ready_for_metadata_creation']}")
print(f"Ready for voiceover:      {script_generation_result['handoff']['ready_for_voiceover']}")
print(f"Wrote:                    {SCRIPT_OUTPUT_DIR}")


In [ ]:
script_generation_result["summary"]


## App 7: Metadata Draft Creator

Creates titles, thumbnail text, description copy, tags, sources, and provisional chapters. Actual chapter timestamps are finalized after voiceover.


In [ ]:
script_result_for_metadata = script_generation_result
if METADATA_SCRIPT_RESULT_PATH:
    external_path = Path(METADATA_SCRIPT_RESULT_PATH)
    script_result_for_metadata = json.loads(external_path.read_text(encoding="utf-8"))
    require_non_empty_script(script_result_for_metadata)
    print(f"Metadata script source: {external_path}")

from src.pipeline.content.metadata_creator import run as run_metadata_creator

metadata_creator_input = build_metadata_creator_input(RUN_ID, script_result_for_metadata, video_packaging_result)
metadata_creator_result = run_metadata_creator(metadata_creator_input)
require_app_ready(metadata_creator_result, "metadata_creator", handoff_key="ready_for_metadata_finalization")
write_json(METADATA_OUTPUT_DIR / "metadata_creator_input.json", metadata_creator_input)
metadata_package = metadata_creator_result["data"]["metadata_package"]
youtube_metadata = metadata_package["youtube_metadata"]

write_json(METADATA_OUTPUT_DIR / "metadata_creator_result.json", metadata_creator_result)
write_json(METADATA_OUTPUT_DIR / "metadata_package.json", metadata_package)
write_json(METADATA_OUTPUT_DIR / "youtube_metadata.json", youtube_metadata)
write_json(METADATA_OUTPUT_DIR / "summary.json", metadata_creator_result["summary"])
write_jsonl(METADATA_OUTPUT_DIR / "llm_calls.jsonl", metadata_creator_result["llm_calls"])

print(f"Metadata status:          {metadata_creator_result['status']}")
print(f"Title options:            {metadata_creator_result['summary']['title_option_count']}")
print(f"Thumbnail options:        {metadata_creator_result['summary']['thumbnail_option_count']}")
print(f"Provisional chapters:     {metadata_creator_result['summary']['chapter_count']}")
print(f"Source URLs:              {metadata_creator_result['summary']['source_url_count']}")
print(f"Ready for finalization:   {metadata_creator_result['handoff']['ready_for_metadata_finalization']}")
print(f"Wrote:                    {METADATA_OUTPUT_DIR}")


In [ ]:
metadata_creator_result["summary"]
